# MPSI reproduction and three-model PM₂.₅ extension

This notebook first reproduces Hafiz's weighted MPSI pipeline exactly, then uses
the same preprocessing and model specification for the agreed three-model
experiment.

**Interpretation boundary:** the outcome is October–February versus
March–September. 

## 1. Method locked to the original repository

- 11 original winter-smog cities for coefficient reproduction
- population Z-scores within each city-year (`ddof=0`)
- October–February label = 1
- effectively unpenalised balanced logistic regression
- five-fold stratified cross-validation grouped by city
- original coefficient-to-weight normalisation
- detection: score ≥ 1.8, at least two satellite Z-scores ≥ 1, and Oct–Feb

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

from src.mpsi_analysis import (
    HAFIZ_CITIES, POLLUTANTS, Z5, Z6,
    load_monthly_panel, add_city_year_zscores,
    fit_formula, add_detection, run_analysis,
)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
BOOTSTRAP_REPS = 2000

## 2. Reproduction

Weekly values are aggregated to city-month means. The original 11-city subset is
then standardised within city-year before fitting the balanced logistic model.

In [ ]:
monthly_raw = load_monthly_panel(DATA_DIR / "AllCities_combined.csv")
monthly = add_city_year_zscores(monthly_raw, POLLUTANTS)

hafiz_sample = monthly[monthly["City"].isin(HAFIZ_CITIES)].copy()
reproduction_fit = fit_formula(hafiz_sample, Z5)
reproduction_detection = add_detection(
    reproduction_fit["data"], Z5,
    reproduction_fit["final_weights"], Z5,
)

print("Rows:", len(reproduction_detection))
print("Cities:", reproduction_detection["City"].nunique())
print("Detected city-months:", int(reproduction_detection["detected"].sum()))
print("Raw coefficients:", reproduction_fit["beta"])
print("Final weights:", reproduction_fit["final_weights"])

Rows: 1056
Cities: 11
Detected city-months: 51
Raw coefficients: [0.6747519384533692, 1.048777913198852, 2.2966646598768645, -1.4150362211551173, -1.1612400240292373]
Final weights: [0.18513151293454186, 0.2877529218335891, 0.6301352822503712, -0.38824311802659295, -0.3186091217780672]


In [ ]:
reproduction_check = pd.read_csv(OUTPUT_DIR / "reproduction_coefficients.csv")
display(reproduction_check.round(6))

          model_id variable  standardised_beta  odds_ratio_per_1SD  final_formula_weight  expected_raw_beta  expected_final_weight
Hafiz reproduction      NO2           0.674752            1.963546              0.185132             0.6748                 0.1851
Hafiz reproduction       CO           1.048778            2.854161              0.287753             1.0488                 0.2878
Hafiz reproduction      SO2           2.296665            9.940971              0.630135             2.2967                 0.6301
Hafiz reproduction       O3          -1.415036            0.242917             -0.388243            -1.4150                -0.3882
Hafiz reproduction     UVAI          -1.161240            0.313098             -0.318609            -1.1612                -0.3186


The reproduced coefficients match the reported vector within 0.001.
This establishes that the original weights came from the documented 11-city,
city-year-standardised pipeline.

## 3. Three-model extension

- **M1:** five satellite indicators, 25 cities, 2,400 city-months
- **M2:** the same five indicators, restricted to the 1,342 PM₂.₅-linked rows
- **M3:** the same 1,342 rows plus PM₂.₅

The satellite Z-scores are defined on the complete satellite panel and retained
when the PM₂.₅ subset is selected. M2 and M3 therefore differ only by the added
PM₂.₅ predictor.

In [ ]:
summary = run_analysis(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    bootstrap_reps=BOOTSTRAP_REPS,
)
print(json.dumps(summary["samples"], indent=2))

{
  "M1_full_5": {
    "rows": 2400,
    "cities": 25
  },
  "M2_subset_5": {
    "rows": 1342,
    "cities": 20
  },
  "M3_subset_6": {
    "rows": 1342,
    "cities": 20
  }
}


## 4. City-held-out performance

All models use five-fold `StratifiedGroupKFold` grouped by city. Model 2 and
Model 3 use the same observations and folds. Confidence intervals use a
city-cluster bootstrap of the out-of-fold predictions.

In [ ]:
performance = pd.read_csv(OUTPUT_DIR / "three_model_performance.csv")
display(performance.round(4))

   model_id  sample_rows  cities            metric  estimate  ci95_low  ci95_high
  M1_full_5         2400      25             AUROC    0.8748    0.8121     0.9228
  M1_full_5         2400      25             AUPRC    0.8375    0.7547     0.9011
  M1_full_5         2400      25 Balanced_accuracy    0.8062    0.7539     0.8516
  M1_full_5         2400      25       Sensitivity    0.7910    0.7330     0.8410
  M1_full_5         2400      25       Specificity    0.8214    0.7664     0.8686
  M1_full_5         2400      25       Brier_score    0.1404    0.1102     0.1765
M2_subset_5         1342      20             AUROC    0.8560    0.7449     0.9367
M2_subset_5         1342      20             AUPRC    0.7924    0.6306     0.9172
M2_subset_5         1342      20 Balanced_accuracy    0.7985    0.7107     0.8691
M2_subset_5         1342      20       Sensitivity    0.7764    0.6855     0.8534
M2_subset_5         1342      20       Specificity    0.8207    0.7307     0.8932
M2_subset_5     

In [ ]:
paired_difference = pd.read_csv(
    OUTPUT_DIR / "M3_minus_M2_paired_differences.csv"
)
display(paired_difference.round(4))

           comparison            metric  estimate  ci95_low  ci95_high
Model 3 minus Model 2             AUROC   -0.0009   -0.0104     0.0073
Model 3 minus Model 2             AUPRC    0.0028   -0.0056     0.0121
Model 3 minus Model 2 Balanced_accuracy    0.0004   -0.0124     0.0141
Model 3 minus Model 2       Sensitivity   -0.0055   -0.0223     0.0122
Model 3 minus Model 2       Specificity    0.0063   -0.0101     0.0205
Model 3 minus Model 2       Brier_score    0.0005   -0.0042     0.0059


## 5. Re-estimated weights

`standardised_beta` is the logistic coefficient per one-SD increase.
`final_formula_weight` applies the original MPSI normalisation: signed absolute
coefficient share followed by unit-SD score scaling.

In [ ]:
coefficient_results = pd.read_csv(
    OUTPUT_DIR / "three_model_coefficients.csv"
)
display(coefficient_results.round(4))

   model_id variable  standardised_beta  odds_ratio_per_1SD  final_formula_weight  beta_ci95_low  beta_ci95_high  weight_ci95_low  weight_ci95_high
  M1_full_5      NO2             0.4933              1.6377                0.2320         0.0129          1.2705           0.0056            0.4653
  M1_full_5       CO             0.5385              1.7135                0.2533         0.2120          0.8583           0.0952            0.3925
  M1_full_5      SO2             1.2727              3.5706                0.5987         0.7615          2.0268           0.3596            0.7748
  M1_full_5       O3            -1.1561              0.3147               -0.5438        -1.5818         -0.8255          -0.6875           -0.3880
  M1_full_5     UVAI            -0.5499              0.5770               -0.2587        -0.9219         -0.2134          -0.3924           -0.0996
M2_subset_5      NO2             0.7675              2.1545                0.3772         0.1561          1.8926

## 6. Detected months under the original threshold

The primary six-variable comparison keeps the gate based on the same five
satellite components. This prevents PM₂.₅ from changing both the score and the
gate simultaneously. A six-component-gate sensitivity result is also exported.

In [ ]:
detection_counts = pd.read_csv(
    OUTPUT_DIR / "three_model_detection_counts.csv"
)
display(detection_counts)

                            model_id  sample_rows  cities  tau                                         gate  detected_city_months
                           M1_full_5         2400      25  1.8 at least two of five satellite Z-scores >= 1                    94
                         M2_subset_5         1342      20  1.8 at least two of five satellite Z-scores >= 1                    48
                         M3_subset_6         1342      20  1.8 at least two of five satellite Z-scores >= 1                    59
M3_subset_6_all_six_gate_sensitivity         1342      20  1.8            at least two of six Z-scores >= 1                    74


In [ ]:
display(Image(filename=str(OUTPUT_DIR / "three_model_roc.png")))
display(Image(filename=str(OUTPUT_DIR / "three_model_weights.png")))